In [1]:
import os
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.v2 as transforms
from torchvision import datasets
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

Usando: cuda


In [2]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

#weights = VGG16_Weights.FIXME
weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\frank/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100%|██████████| 528M/528M [00:07<00:00, 72.8MB/s] 


In [3]:
# Freeze base model
vgg_model.requires_grad_(False)
next(iter(vgg_model.parameters())).requires_grad

False

In [4]:
# Add Layers to Mode
vgg_model.classifier[0:3]

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
)

In [5]:
#N_CLASSES: buildings, forest, glacier, mountain, sea, street = FIXME
N_CLASSES = 6

my_model = nn.Sequential(
    vgg_model.features,
    vgg_model.avgpool,
    nn.Flatten(),
    vgg_model.classifier[0:3],
    nn.Linear(4096, 500),
    nn.ReLU(),
    nn.Linear(500, N_CLASSES)
)
total     = sum(p.numel() for p in my_model.parameters())
trainable = sum(p.numel() for p in my_model.parameters() if p.requires_grad)
print(f'Parámetros totales:      {total:,}')
print(f'Parámetros entrenables:  {trainable:,}')
print(f'Parámetros congelados:   {total - trainable:,}')
my_model

Parámetros totales:      119,530,738
Parámetros entrenables:  2,051,506
Parámetros congelados:   117,479,232


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [6]:
# Compile model
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(filter(lambda p: p.requires_grad, my_model.parameters()))
my_model = my_model.to(device)

In [7]:
# Data transforms
pre_trans = weights.transforms()

In [8]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

train_tf = transforms.Compose([
    transforms.Resize((IMG_WIDTH, IMG_HEIGHT)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_WIDTH, IMG_HEIGHT)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

c:\Users\frank\anaconda3\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [9]:
# Load data
base_path  = os.path.join(os.path.dirname(os.path.abspath('quiz_2.ipynb')), 'data')
train_path = os.path.join(base_path, 'seg_train')
val_path   = os.path.join(base_path, 'seg_test')

train_dataset_full = datasets.ImageFolder(root=train_path, transform=train_tf)
val_dataset_full   = datasets.ImageFolder(root=val_path,   transform=val_tf)

print(f'Clases detectadas: {train_dataset_full.classes}')
print(f'Total train: {len(train_dataset_full):,} imágenes')
print(f'Total val:   {len(val_dataset_full):,} imágenes')

Clases detectadas: ['seg_train']
Total train: 14,034 imágenes
Total val:   3,000 imágenes


In [10]:
targets     = np.array(val_dataset_full.targets)
sample_size = 100

splitter = StratifiedShuffleSplit(n_splits=2, test_size=sample_size, random_state=42)
sample_datasets = []
for _, sample_idx in splitter.split(np.zeros(len(targets)), targets):
    sample_datasets.append(Subset(val_dataset_full, sample_idx))

val_dataset  = sample_datasets[0]
test_dataset = sample_datasets[1]

print(f'Dataset de validación:     {len(val_dataset)} imágenes')
print(f'Dataset de prueba interno: {len(test_dataset)} imágenes')

Dataset de validación:     100 imágenes
Dataset de prueba interno: 100 imágenes


In [11]:
torch.backends.cudnn.benchmark = True
nw = min(4, os.cpu_count())

train_loader = DataLoader(train_dataset_full, batch_size=64, shuffle=True,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)
valid_loader = DataLoader(val_dataset,        batch_size=64, shuffle=False,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset,       batch_size=64, shuffle=False,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)

train_N = len(train_loader.dataset)
valid_N = len(valid_loader.dataset)
test_N  = len(test_loader.dataset)

for images, labels in train_loader:
    print(f'Batch shape: {images.shape}')  # Debe ser [64, 3, 224, 224]
    break

Batch shape: torch.Size([64, 3, 224, 224])


In [12]:
# Train Model

# Experimento #1
def train(model, loader, N, optimizer, loss_fn, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in tqdm(loader, desc='  train', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += out.argmax(1).eq(y).sum().item()
    print(f'  Train | Loss: {total_loss/len(loader):.4f} | Acc: {correct/N:.4f}')
    return correct / N


def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out        = model(x)
            total_loss += loss_fn(out, y).item()
            correct    += out.argmax(1).eq(y).sum().item()
            total      += y.size(0)
    print(f'  Val   | Loss: {total_loss/len(loader):.4f} | Acc: {correct/total:.4f}')
    return correct / total

In [13]:
epochs = 15

train_accs_exp1 = []
val_accs_exp1   = []
best_val_exp1   = 0.0

print('=== EXPERIMENTO 1: Solo última capa (modelo congelado) ===')
for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    tr_acc  = train(my_model, train_loader, train_N, optimizer, loss_function, device)
    val_acc = validate(my_model, valid_loader, loss_function, device)
    train_accs_exp1.append(tr_acc)
    val_accs_exp1.append(val_acc)
    if val_acc > best_val_exp1:
        best_val_exp1 = val_acc
        torch.save(my_model.state_dict(), 'best_exp1.pth')

print(f'\nMejor Val Acc Experimento 1: {best_val_exp1:.4f}')
my_model.load_state_dict(torch.load('best_exp1.pth', weights_only=True))

=== EXPERIMENTO 1: Solo última capa (modelo congelado) ===
Epoch: 0


  Train | Loss: 0.0082 | Acc: 0.9959
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 1


  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 2


  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 3


  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 4


...

  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 7


  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 8


  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 9


  Train | Loss: 0.0000 | Acc: 1.0000
  Val   | Loss: 0.0000 | Acc: 1.0000
Epoch: 10


KeyboardInterrupt: 

In [ ]:
# Unfreeze the base model
vgg_model.requires_grad_(True)
optimizer = Adam(my_model.parameters(), lr=.0001)

In [ ]:
epochs = 5

train_accs_exp2 = []
val_accs_exp2   = []
best_val_exp2   = 0.0

print('=== EXPERIMENTO 2: Fine-tuning (modelo descongelado) ===')
for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    tr_acc  = train(my_model, train_loader, train_N, optimizer, loss_function, device)
    val_acc = validate(my_model, valid_loader, loss_function, device)
    train_accs_exp2.append(tr_acc)
    val_accs_exp2.append(val_acc)
    if val_acc > best_val_exp2:
        best_val_exp2 = val_acc
        torch.save(my_model.state_dict(), 'best_exp2.pth')

print(f'\nMejor Val Acc Experimento 2: {best_val_exp2:.4f}')
my_model.load_state_dict(torch.load('best_exp2.pth', weights_only=True))

In [ ]:
# Evalue the Model
print('=== EVALUACIÓN FINAL EN TEST SET ===')

my_model.load_state_dict(torch.load('best_exp1.pth', weights_only=True))
print('\nExperimento 1 (solo última capa):')
test_acc_exp1 = validate(my_model, test_loader, loss_function, device)

my_model.load_state_dict(torch.load('best_exp2.pth', weights_only=True))
print('\nExperimento 2 (fine-tuning):')
test_acc_exp2 = validate(my_model, test_loader, loss_function, device)

In [ ]:
# Gráfica 1: curvas de accuracy por época
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, 16), train_accs_exp1, 'b--', label='Train Exp1')
axes[0].plot(range(1, 16), val_accs_exp1,   'b-',  label='Val Exp1')
axes[0].plot(range(16, 21), train_accs_exp2, 'r--', label='Train Exp2 (fine-tune)')
axes[0].plot(range(16, 21), val_accs_exp2,   'r-',  label='Val Exp2 (fine-tune)')
axes[0].axvline(x=15.5, color='gray', linestyle=':', label='Inicio fine-tuning')
axes[0].set_title('Accuracy por Época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Gráfica 2: comparación con MLP Lab 1
mlp_val_acc = 0.64  

models  = ['MLP Lab 1\n(sin CNN)', 'Exp1\n(congelado)', 'Exp2\n(fine-tune)']
val_bar = [mlp_val_acc, best_val_exp1, best_val_exp2]
colors  = ['#aaaaaa', 'steelblue', 'tomato']

axes[1].bar(models, val_bar, color=colors)
axes[1].set_title('Mejor Val Accuracy – Comparación')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
for i, v in enumerate(val_bar):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Transfer Learning vs MLP Lab 1 – Intel Image Classification', fontsize=13)
plt.tight_layout()
plt.show()